In [16]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)

# 02 编码器模块 (core.encoders)

提供各类特征编码功能，所有编码器均遵循 sklearn Transformer 接口规范。

**支持的编码器：**\n- `WOEEncoder`: 证据权重编码（风控专用）\n- `TargetEncoder`: 目标编码\n- `CountEncoder`: 计数编码\n- `OneHotEncoder`: 独热编码\n- `OrdinalEncoder`: 序数编码\n- `QuantileEncoder`: 分位数编码\n- `CatBoostEncoder`: CatBoost编码\n- `GBMEncoder`: 梯度提升树编码器

**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [17]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from hscredit import init_setting
from hscredit.core.encoders import (
    WOEEncoder, TargetEncoder, CountEncoder, OneHotEncoder
)

init_setting()

# 加载数据
_roots = [Path.cwd(), Path.cwd() / 'examples', Path.cwd().parent]
_fp = None
for _r in _roots:
    for _n in ('hscredit_yyp.xlsx', 'hengshucredit_yyp.xlsx'):
        if (_r / _n).is_file():
            _fp = _r / _n
            break
    if _fp is not None:
        break
if _fp is None:
    raise FileNotFoundError('请将 hscredit_yyp.xlsx 放在 examples/')

df = pd.read_excel(_fp)

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

# 重命名为英文列名
df = df.rename(columns={
    '中智小牛分C3': 'score_c3',
    '珊瑚92': 'score_coral',
    '极光欺诈分6v1': 'score_fraud',
    '轻花老客海纳子分V1': 'score_haina',
})

print(f"样本数: {len(df):,}")
print(f"坏样本率: {df['target'].mean():.2%}")

样本数: 970
坏样本率: 16.70%


## 1. 先进行分箱，再WOE编码

In [18]:
from hscredit.core.binning import OptimalBinning

# 对连续分数字段先分箱，再作为类别特征进行后续编码
raw_features = ['score_c3', 'score_fraud', 'score_haina']
X_raw = df[raw_features].fillna(df[raw_features].median())

binner = OptimalBinning(method='quantile', max_n_bins=5, min_bin_size=0.05)
df_binned_raw = binner.fit_transform(X_raw, df['target']).astype(str)
df_binned_raw.columns = [f'{c}_bin' for c in df_binned_raw.columns]

print(f"分箱后的列: {df_binned_raw.columns.tolist()}")
print("\n分箱结果预览:")
display(df_binned_raw.head())

分箱后的列: ['score_c3_bin', 'score_fraud_bin', 'score_haina_bin']

分箱结果预览:


,score_c3_bin,score_fraud_bin,score_haina_bin
0,1,0,1
1,1,0,1
2,1,0,1
3,1,0,1
4,1,0,1


In [19]:
# WOE编码 - 对分箱后的类别特征进行编码
woe_encoder = WOEEncoder()
woe_encoder.fit(df_binned_raw, df['target'])
df_woe = woe_encoder.transform(df_binned_raw.copy())

print("WOE编码结果预览:")
display(df_woe.head())

WOE编码结果预览:


,score_c3_bin,score_fraud_bin,score_haina_bin
0,0.0573,-0.0388,-0.2970
1,0.0573,-0.0388,-0.2970
2,0.0573,-0.0388,-0.2970
3,0.0573,-0.0388,-0.2970
4,0.0573,-0.0388,-0.2970


## 2. Target编码（建议先对连续分数字段离散化）

连续高基数数值列直接做目标编码容易不稳定。对风控分数字段，更推荐先分箱，再对箱标签做 TargetEncoder。

In [20]:
# 对分箱后的类别特征进行Target编码
target_encoder = TargetEncoder(
    cols=['score_c3_bin'],
    smoothing=10
)
target_encoder.fit(df_binned_raw[['score_c3_bin']], df['target'])
df_target_demo = df_binned_raw[['score_c3_bin']].copy()
df_target_demo['score_c3_bin_target'] = target_encoder.transform(
    df_target_demo[['score_c3_bin']]
)['score_c3_bin']

print("Target编码结果预览:")
display(df_target_demo.head(10))

Target编码结果预览:


,score_c3_bin,score_c3_bin_target
0,1,0.1755
1,1,0.1755
2,1,0.1755
3,1,0.1755
4,1,0.1755
5,1,0.1755
6,1,0.1755
7,1,0.1755
8,1,0.1755
9,1,0.1755


## 3. Count编码（计数编码）

In [21]:
# Count编码
count_encoder = CountEncoder(
    cols=['score_c3_bin'],
    normalize=False
)
count_encoder.fit(df_binned_raw[['score_c3_bin']])
df_count_demo = df_binned_raw[['score_c3_bin']].copy()
df_count_demo['score_c3_bin_count'] = count_encoder.transform(
    df_count_demo[['score_c3_bin']]
)['score_c3_bin']

print("Count编码结果预览:")
display(df_count_demo.head(10))

Count编码结果预览:


,score_c3_bin,score_c3_bin_count
0,1,672
1,1,672
2,1,672
3,1,672
4,1,672
5,1,672
6,1,672
7,1,672
8,1,672
9,1,672


## 4. Pipeline中使用编码器（推荐做法：先分箱，再编码）

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from hscredit.core.models import LogisticRegression

# 准备数据：连续分数字段先分箱，再对分箱标签做Target编码
feature_cols = ['score_c3_bin', 'score_fraud_bin', 'score_haina_bin']
X = df_binned_raw[feature_cols].copy()
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 使用TargetEncoder + LR的Pipeline
pipeline = Pipeline([
    ('target_encoder', TargetEncoder(cols=feature_cols, smoothing=10)),
    ('classifier', LogisticRegression(max_iter=1000))
])

# 训练
pipeline.fit(X_train, y_train)

# 预测
y_prob = pipeline.predict_proba(X_test)
if y_prob.ndim == 2:
    y_prob = y_prob[:, 1]

# 评估
from hscredit.core.metrics import ks, auc
print(f"使用特征: {feature_cols}")
print(f"测试集KS: {ks(y_test, y_prob):.4f}")
print(f"测试集AUC: {auc(y_test, y_prob):.4f}")

使用特征: ['score_c3_bin', 'score_fraud_bin', 'score_haina_bin']
测试集KS: 0.2902
测试集AUC: 0.6540
